# 00 · Setup and connect

Install the SDK, authenticate, and read the **event clock**. So you know which phase is
running and what you should be doing right now. Two minutes, no model training.

The event runs as a **timed build-and-validate phase** followed by a **sequence of sealed
rounds**. How long each is, and how many rounds there are, is set per event and read off
`cadence.build_phase_minutes`, `cadence.round_minutes` and `cadence.n_rounds` - the staging
event these notebooks were checked against ran 35 minutes then four rounds of 7, another may
give you hours. The cells below print your event's real clock; plan against that, never against
a number from a write-up. Only the round currently open is being
scored, and per-round scores accumulate into the cumulative standings that decide a
display-only event (a money event is decided by the final stake balance instead).
Full description in the [README](../README.md#the-event).

**Next:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) →
[`02_train_and_submit.ipynb`](02_train_and_submit.ipynb)

## 1. Install

In [ ]:
# everestapi is the Everesteer SDK. This notebook only needs the SDK and a parquet reader.
# Install only what is missing. Re-running this cell on a machine that already has
# the SDK should not be a network round-trip that tries to upgrade it.
try:
    import everestapi, pandas, pyarrow  # noqa: F401
    assert tuple(map(int, everestapi.__version__.split(".")[:3])) >= (0, 3, 32)
except (ImportError, AssertionError, ValueError):
    %pip install --quiet "everestapi>=0.3.32" pandas pyarrow

## 2. Authenticate

Credentials come from onboarding's **Copy setup command** (*Install & connect your agent → Step 2*).
Set them in your shell **before** launching Jupyter (or in a local `.env`). Never commit them:

`EIQ_API_KEY`, `EIQ_BASE_URL` (defaults to `https://app.everesteer.ai`)

`CF_ACCESS_CLIENT_ID` / `CF_ACCESS_CLIENT_SECRET` are only needed against the gated
**staging** environment. Omit them on the public site.

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://app.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")

# Only the gated STAGING mirror needs a Cloudflare Access service token; the public
# site does not. The SDK picks the CF_ACCESS_* env vars up automatically when set.
if "staging" in base_url and not (
    os.environ.get("CF_ACCESS_CLIENT_ID") and os.environ.get("CF_ACCESS_CLIENT_SECRET")
):
    raise RuntimeError(
        "Staging is behind Cloudflare Access - set CF_ACCESS_CLIENT_ID / "
        "CF_ACCESS_CLIENT_SECRET, or point EIQ_BASE_URL at the public site."
    )

client = EverestAPI(api_key=api_key, base_url=base_url)
# Only `status` is meaningful here. The health probe is a constant-cost liveness
# check, so its `db` and `dataset` fields are always null by design.
print("connection:", client.health().get("status"))  # -> connection: ok

## 3. Where are we in the event?

`get_started` is mode-aware: it reports the shape of *your* event, including a `cadence`
object when the event runs on a clock. **Read the phase, never assume it**, round lengths
are configured per event and can be paused or extended on the day.

The cell below turns that object into a short readout: where you are, what to do
right now, and whether money is on the line.

In [ ]:
from datetime import datetime, timezone

started = client.get_started() or {}
cadence = started.get("cadence") or {}
# get_started's event_staking block is the ONLY authority on whether this event carries
# money (see AGENTS.md#event-staking). Branch on `money_event`, NOT on the block being
# present: a display-only event ships the block too, with money_event false.
event_staking = started.get("event_staking") or {}
is_money = bool(event_staking.get("money_event"))


def line(label, text):
    """One labelled row; continuation rows pass label=''."""
    print(f"  {label:<7} {text}")


def when(ts):
    """ISO timestamp -> 'YYYY-MM-DD HH:MM UTC', or the raw value if unparseable."""
    try:
        return datetime.fromisoformat(ts).astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    except (TypeError, ValueError):
        return str(ts)


print(f"scope:  {started.get('mode') or started.get('scope') or 'unknown'}")
print("money:  " + ("YES - USDC is staked, so the FINAL STAKE BALANCE is the result"
                    if is_money else "no - display-only; the cumulative standings are the result"))

if not cadence:
    print("\nNo cadence on this key - either a legacy no-clock event, or a tournament key.")
else:
    phase = cadence.get("phase")
    open_window = cadence.get("open_window")
    n_rounds = cadence.get("n_rounds")
    done = phase in ("done", "complete") or started.get("event_status") == "finished"
    print()

    # --- where you are, and what it means you should be doing -------------------
    if done:
        line("WHERE", f"Event COMPLETE - all {n_rounds} rounds are scored and settled.")
        if started.get("ends_at"):
            line("", f"Ended {when(started['ends_at'])}.")
        if is_money:
            line("RESULT", "Your final recorded STAKE BALANCE is the outcome, not the points board.")
            line("DO NOW", "get_event_staking()         - the money view: windows[].settlements")
            line("", "get_diagnostics_standings() - the points view of the same race")
        else:
            line("RESULT", "The cumulative standings are final.")
            line("DO NOW", "get_diagnostics_standings() - the final result")
    elif phase == "build":
        line("WHERE", "BUILD & VALIDATE - nothing here counts toward the standings yet.")
        line("DO NOW", 'download_dataset(split="train") -> fit -> rehearse the upload path')
        line("", "with submit_validation_diagnostics() on the practice board.")
    elif open_window:
        rnd = open_window.replace("round_", "")
        line("WHERE", f"ROUND {rnd} OF {n_rounds} IS OPEN ({open_window}) - this one is being scored.")
        line("DO NOW", 'download_dataset(split="live") -> predict -> submit_event_predictions(...)')
        if is_money and event_staking.get("draft_window"):
            line("MONEY", f"Stake drafts are OPEN for {event_staking['draft_window']} and lock "
                         f"when it closes:")
            line("", 'set_stake_allocation(model, amount_usdc="...", '
                     f'window="{event_staking["draft_window"]}")')
    else:
        # Between rounds (or a stake phase): no round is scoreable, the practice board is open.
        line("WHERE", f"BETWEEN ROUNDS (phase {phase!r}) - no round is open to submit into.")
        line("DO NOW", "Wait for the next round; keep a fitted model ready to predict with.")
        line("", "The practice board still takes submit_validation_diagnostics() uploads.")

    # --- the clock --------------------------------------------------------------
    secs = cadence.get("seconds_until_next_phase")
    clock = []
    if secs is not None:
        clock.append(f"next phase in {int(secs) // 60} min {int(secs) % 60} s"
                     + (f" (at {when(cadence['phase_ends_at'])})" if cadence.get("phase_ends_at") else ""))
    if cadence.get("clock_paused"):
        clock.append("! the clock is PAUSED - phase lengths on the day are not the plan")
    for i, bit in enumerate(clock):
        line("CLOCK" if i == 0 else "", bit)

    # --- progress through the event, one line instead of the raw phase list -----
    seq = cadence.get("phase_sequence") or []
    if seq:
        track, seen = [], set()
        for p in seq:
            slot = "build" if p == "build" else "done" if p == "done" else \
                   "R" + p.replace("round_", "").split("_")[0]
            if slot not in seen:          # collapse round_N_opening/round_N/round_N_closing
                seen.add(slot)
                here = slot == ("build" if phase == "build" else "done" if done else
                                "R" + str(phase or "").replace("round_", "").split("_")[0])
                track.append(f"[{slot}]" if here else slot)
        line("TRACK", " > ".join(track))

    # --- only warn about the fence while it is something you can wait out -------
    # intake_fenced fences ROUND submissions only; it is also true in build and between
    # rounds, where the practice board is open regardless.
    if cadence.get("intake_fenced"):
        if done:
            line("NOTE", "Intake is closed for good now that the event has finished.")
        elif open_window:
            line("NOTE", "! the round is opening or closing - round submissions are refused now.")
    if cadence.get("diagnostics_maintenance") and not done:
        line("NOTE", "! the practice board is briefly down for maintenance - retry later.")

## 5. What you have to spend

Three budgets, all fixed for the whole event:

- **Round uploads** come from one per-event pool. Every model and every round draws on the
  same allowance and it never refills, so budget across the whole event rather than spending
  it on round-1 experiments. Practice-board uploads are free and do not draw on it. (A
  skipped round is not scored as a zero - the standings take the mean of the rounds you *did*
  score - but it is still a round you cannot be paid for or raise that mean with.)
- **Compute credits** fund hosted training (`client.train(...)`). The grant below is the entire
  hosted budget: hackathon keys cannot buy more, and it is spendable immediately, with nothing
  locked pending funding. Fitting on your own hardware costs none of it.
- **The hosted-LLM wallet** funds inference through the platform's Claude/OpenAI gateway
  (`/api/v1/llm/`), and is separate from compute credits. It has *two* limits - a balance and a
  rolling-24h cap - and whichever binds first stops you.

In [ ]:
def plural(n, word):
    return f"{n} {word}" + ("" if n == 1 else "s")


status = {}
try:
    status = client.get_status() or {}
    left = status.get("uploads_remaining")
    print(f"uploads:  {left if left is not None else 'uncapped'} left in this event's round-upload pool"
          + (f", shared across all {cadence.get('n_rounds')} rounds" if cadence.get("n_rounds") else ""))
except Exception as e:
    print("uploads:  get_status unavailable:", e)

try:
    credits = client.get_compute_credits() or {}
    avail = (credits.get("available_cents") or 0) / 100
    held = (credits.get("reserved_cents") or 0) / 100
    print(f"compute:  ${avail:,.2f} available"
          + (f", ${held:,.2f} held by running jobs" if held else "")
          + f"  (${credits.get('total_spent_usd', 0):,.2f} spent so far)")

    # What that balance actually buys, at this event's posted rates.
    rates = started.get("compute_rate_card_usd_per_hr") or {}
    if avail and rates:
        buys = [f"{avail / r:.1f} h on {tier} (${r}/h)"
                for tier, r in sorted(rates.items(), key=lambda kv: kv[1])[:2]]
        print(f"          about {', or '.join(buys)}")
    # Pre-funded means spendable right now - nothing is locked pending funding. Hackathon
    # keys cannot buy more, so this grant is the whole hosted-compute budget.
    print(f"          pre-funded for this event: {started.get('hosted_train_funded', 'not reported')}")
except Exception as e:
    print("compute:  get_compute_credits unavailable:", e)

# The hosted-LLM wallet, i.e. the gateway that fronts Claude and OpenAI. A separate
# budget from compute credits. The SDK has no wrapper for it yet, so call the endpoint
# through the client: that reuses the same auth and Cloudflare Access headers as
# every other call in this notebook.
try:
    llm = client._request("GET", "/api/v1/llm/me") or {}
    bal = (llm.get("balance_cents") or 0) / 100
    held = (llm.get("reserved_cents") or 0) / 100
    cap = llm.get("daily_cap_cents")
    print(f"llm:   ${bal:,.2f} available"
          + (f", ${held:,.2f} held in flight" if held else "")
          + f"  (${(llm.get('total_spent_cents') or 0) / 100:,.2f} spent so far)")
    # Two independent limits: the wallet, and a rolling-24h cap. Whichever binds first stops you.
    if cap:
        print(f"          ${(llm.get('daily_spent_cents') or 0) / 100:,.2f} of a ${cap / 100:,.2f}"
              f" daily cap used ({llm.get('daily_cap_scope', 'unknown')} scope)")
    tok = llm.get("tokens") or {}
    print(f"          {plural(llm.get('requests_total', 0), 'request')} lifetime,"
          f" {llm.get('requests_today', 0)} today"
          f" | tokens in {tok.get('input', 0):,} / out {tok.get('output', 0):,}")
except Exception as e:
    # A data-only key carries no LLM scope; that is not a setup problem.
    print("llm:   hosted-LLM wallet unavailable on this key:", e)

models = status.get("models") or []
print(f"\nmodels:   {len(models)} registered"
      + (f": {', '.join(models)}" if models else " - create_model(...) before your first submit"))
print("next:    ", ", ".join(started.get("next_actions", [])) or "none reported")

## You're connected

- Green `health()`, a phase printed above, and a schema you can read → you're set up.
- **Now:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) to see what you're modelling.
- **Then:** [`02_train_and_submit.ipynb`](02_train_and_submit.ipynb) to fit a baseline and enter a round.

Agents driving the tools directly should read [`AGENTS.md`](../AGENTS.md) instead. It carries
the full loop, the staking surface, and the research skills.